In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [10]:
data = pd.read_parquet('yellow_tripdata_2024-01.parquet')
data.to_csv('yellow_tripdata_2024-01.csv', index=False)

In [11]:
data = pd.read_parquet('yellow_tripdata_2024-02.parquet')
data.to_csv('yellow_tripdata_2024-02.csv', index=False)

In [12]:
data = pd.read_parquet('yellow_tripdata_2024-03.parquet')
data.to_csv('yellow_tripdata_2024-03.csv', index=False)

In [15]:
# List of CSV files
csv_files = ['yellow_tripdata_2024-01.csv', 'yellow_tripdata_2024-02.csv', 'yellow_tripdata_2024-03.csv']
# Create a list to store DataFrames
dfs = []
# Read each CSV file and append to the list
for file in csv_files:
   dfs.append(pd.read_csv(file))
# Concatenate the list of DataFrames
df_res = pd.concat(dfs, ignore_index=True)
print(df_res)

C:\Users\HP\AppData\Local\Temp\ipykernel_23928\1735401107.py:7: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))
C:\Users\HP\AppData\Local\Temp\ipykernel_23928\1735401107.py:7: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))
C:\Users\HP\AppData\Local\Temp\ipykernel_23928\1735401107.py:7: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))


         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0               2  2024-01-01 00:57:55   2024-01-01 01:17:43              1.0   
1               1  2024-01-01 00:03:00   2024-01-01 00:09:36              1.0   
2               1  2024-01-01 00:17:06   2024-01-01 00:35:01              1.0   
3               1  2024-01-01 00:36:38   2024-01-01 00:44:56              1.0   
4               1  2024-01-01 00:46:51   2024-01-01 00:52:57              1.0   
...           ...                  ...                   ...              ...   
9554773         2  2024-03-31 23:16:45   2024-03-31 23:29:20              NaN   
9554774         1  2024-03-31 23:29:28   2024-03-31 23:43:47              NaN   
9554775         2  2024-03-31 23:15:00   2024-03-31 23:47:29              NaN   
9554776         2  2024-03-31 23:27:53   2024-03-31 23:45:44              NaN   
9554777         2  2024-03-31 23:10:50   2024-03-31 23:31:59              NaN   

         trip_distance  Rat

In [16]:
df_res.to_csv('combined_yellow_tripdata.csv', index=False)

In [17]:
df = df_res.copy()  # keep the original combined data untouched
initial_rows = len(df)
print(f"Starting rows: {initial_rows:,}")

Starting rows: 9,554,778


In [2]:
input_file = 'combined_yellow_tripdata.csv' 
output_file = 'cleaned_yellow_tripdata.csv'
report_file = 'cleaning_report.txt'

# We'll keep our 'issues' list here to track what gets deleted across all cells
issues = []

In [34]:

print(f"Loading data from: {input_file}...")

# Read the CSV into our DataFrame (table) named 'df'
df = pd.read_csv(input_file, low_memory=False)

original_count = len(df)
print(f"Original records loaded: {original_count:,}")

# Let's peek at the first 5 rows to see what it looks like before cleaning!
df.head()

Loading data from: combined_yellow_tripdata.csv...
Original records loaded: 9,554,778


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [36]:
# Drop Missing Values 

critical_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 
                 'passenger_count', 'trip_distance', 'fare_amount', 
                 'PULocationID', 'DOLocationID', 'payment_type']

missing_before = len(df)
df = df.dropna(subset=[c for c in critical_cols if c in df.columns])
missing_dropped = missing_before - len(df)

issues.append(f"Missing critical values dropped: {missing_dropped:,}")
print(f"Dropped {missing_dropped:,} rows with missing critical data.")
print(f"Rows remaining: {len(df):,}")

Dropped 751,962 rows with missing critical data.
Rows remaining: 8,802,816


In [38]:
# Passengers, Distance & Fare

# 1. Fix Passengers
if 'passenger_count' in df.columns:
    invalid_pass = len(df[(df['passenger_count'] < 1) | (df['passenger_count'] > 9)])
    df = df[(df['passenger_count'] >= 1) & (df['passenger_count'] <= 9)]
    issues.append(f"Invalid passenger_count dropped: {invalid_pass:,}")

# 2. Fix Distance
if 'trip_distance' in df.columns:
    invalid_dist = len(df[df['trip_distance'] <= 0])
    df = df[df['trip_distance'] > 0]
    issues.append(f"Zero/negative distance dropped: {invalid_dist:,}")

# 3. Fix Fares
if 'fare_amount' in df.columns:
    invalid_fare = len(df[(df['fare_amount'] <= 0) | (df['fare_amount'] > 1000)])
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] <= 1000)]
    issues.append(f"Invalid fare_amount dropped: {invalid_fare:,}")

print(f"Rows remaining after numeric filters: {len(df):,}")

Rows remaining after numeric filters: 8,479,648


In [39]:
# Fix Dates and Durations

if 'tpep_pickup_datetime' in df.columns and 'tpep_dropoff_datetime' in df.columns:
    # Convert text dates to computer dates
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')

    # Drop failed conversions
    invalid_ts = len(df[df['tpep_pickup_datetime'].isna() | df['tpep_dropoff_datetime'].isna()])
    df = df[df['tpep_pickup_datetime'].notna() & df['tpep_dropoff_datetime'].notna()]
    issues.append(f"Invalid timestamps dropped: {invalid_ts:,}")

    # Calculate trip duration in minutes
    df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

    # Drop impossible times (negative time, or longer than 24 hours)
    invalid_dur = len(df[(df['trip_duration_min'] <= 0) | (df['trip_duration_min'] > 1440)])
    df = df[(df['trip_duration_min'] > 0) & (df['trip_duration_min'] <= 1440)]
    issues.append(f"Impossible duration dropped: {invalid_dur:,}")

print(f"Rows remaining after date/time filters: {len(df):,}")

Rows remaining after date/time filters: 8,479,451


In [40]:
# Duplicates and Locations

# Drop exact duplicates
dup_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 
            'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount']
dup_cols = [c for c in dup_cols if c in df.columns]

if dup_cols:
    duplicates = df.duplicated(subset=dup_cols, keep='first').sum()
    df = df.drop_duplicates(subset=dup_cols, keep='first')
    issues.append(f"Duplicate records dropped: {duplicates:,}")

# Drop invalid locations
for col in ['PULocationID', 'DOLocationID']:
    if col in df.columns:
        invalid_loc = len(df[df[col] <= 0])
        df = df[df[col] > 0]
        issues.append(f"Invalid {col} dropped: {invalid_loc:,}")

cleaned_count = len(df)
print(f"Cleaning complete! Final record count: {cleaned_count:,}")

Cleaning complete! Final record count: 8,479,450


In [41]:
df.head(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_min
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.00,19.800000
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.00,6.600000
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.00,17.916667
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.00,8.300000
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.00,6.100000
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1.0,4.70,1.0,N,148,141,1,29.6,3.5,0.5,6.90,0.0,1.0,41.50,2.5,0.00,32.383333
6,2,2024-01-01 00:49:44,2024-01-01 01:15:47,2.0,10.82,1.0,N,138,181,1,45.7,6.0,0.5,10.00,0.0,1.0,64.95,0.0,1.75,26.050000
8,2,2024-01-01 00:26:01,2024-01-01 00:54:12,1.0,5.44,1.0,N,161,261,2,31.0,1.0,0.5,0.00,0.0,1.0,36.00,2.5,0.00,28.183333
9,2,2024-01-01 00:28:08,2024-01-01 00:29:16,1.0,0.04,1.0,N,113,113,2,3.0,1.0,0.5,0.00,0.0,1.0,8.00,2.5,0.00,1.133333
10,2,2024-01-01 00:35:22,2024-01-01 00:41:41,2.0,0.75,1.0,N,107,137,1,7.9,1.0,0.5,0.00,0.0,1.0,12.90,2.5,0.00,6.316667


In [42]:
# Save the clean table to a new CSV file

df.to_csv(output_file, index=False)
print(f"Cleaned data saved to {output_file}!")

# Print out the summary of everything we did
print("\n--- SUMMARY OF ISSUES FIXED ---")
for issue in issues:
    print(f" • {issue}")

Cleaned data saved to cleaned_yellow_tripdata.csv!

--- SUMMARY OF ISSUES FIXED ---
 • Missing critical values dropped: 751,962
 • Invalid passenger_count dropped: 105,931
 • Zero/negative distance dropped: 110,832
 • Invalid fare_amount dropped: 106,405
 • Invalid timestamps dropped: 0
 • Impossible duration dropped: 197
 • Duplicate records dropped: 1
 • Invalid PULocationID dropped: 0
 • Invalid DOLocationID dropped: 0
